# Spark Declarative Pipeline

## 1. Spark Declarative Pipeline Overview

### 1. Introducing Declarative Pipelines: Reliable data pipelines made easy
**Simplified Pipeline Authoring**
* **Write SQL or Python**
* **No orchestration logic**: Lakeflow manages the execution plan for you
* **Zero boilerplate**: Error handling & dependency management built-in

**Intelligent Optimization at Scale**
* **Auto-scaling**: Pipelines scale automatically as data volumes grow
* **Self-healing**: Automatic recovery from failures — no manual steps
* **Lower overhead**: Better reliability with much less operational effort

**Unified Batch and Streaming**
- **Process historical and real-time data seamlessly**
- **Adapts and optimizes** for performance and cost
- **Same code handles both batch and streaming**


### 2. Creating a Spark Declarative Pipeline

**Create an ETL Pipeline — From the Workspace Menu**
* From anywhere in your workspace, click the options (⋮) button in the navigation bar.
* Select Create from the context menu.
* Then choose ETL Pipeline from the submenu.
![image_1794907298312.png](./image_1794907298312.png "image_1794907298312.png")

**Create an ETL Pipeline — From Jobs & Pipelines**
* Go directly to the Jobs & Pipelines section in the main navigation bar.
* Click the Create button in the top-right corner.
* Then select ETL Pipeline — Build and run an ETL pipeline with SQL and Python.
![image_1794907424481.png](./image_1794907424481.png "image_1794907424481.png")

## 2. Build Data Pipelines with Apache Spark Declarative Pipelines 

### 1. Dataset Types 

#### Streaming Table

* Streaming Tables are designed specifically for streaming or incremental data processing.
* Increases efficiency and decreases cost.
* Incremental Processing Modes: Supports batch or streaming for incremental data processing (exactly-once processing of files)
* SQL syntax to create streaming tables: CREATE OR REFRESH STREAMING TABLE
* Must use FROM STREAM read_files() to enable incremental streaming reads with checkpointing.
* This syntax leverages Databricks Auto Loader, which automatically tracks new files and ensures reliable, incremental ingestion.
* No Duplicate Reads: File names are guaranteed to be read only once.

Syntax:
```
CREATE OR REFRESH STREAMING TABLE table_name
AS
SELECT ...
FROM STREAM source_table;

#### Materialized Views

* Materialized Views are used to store the results of SQL queries physically.
* When upstream data changes, the Materialized View is refreshed to reflect the latest data.
* Pipeline integration: Automatically maintained by the pipeline
* Refresh mechanism: CREATE OR REFRESH MATERIALIZED VIEW
* Flexible Pipeline Placement: Can be used anywhere in your pipeline, not just in the gold layer.
* Incremental Refresh Capability: Where applicable, results are incrementally refreshed, avoiding a full rebuild when new data arrives. Supported on **Serverless compute**.
* Cost-Based Optimization: Incremental refresh is driven by a cost-based optimizer to power fast and efficient transformations on Serverless compute.
* They are particularly useful for:
    - Aggregations and analytical transformations.
    - Preparing Gold-layer datasets.
    - Complex SQL transformations.
    - Improving query performance.
    - Creating reusable, precomputed datasets.


Syntax:
```
CREATE OR REFRESH MATERIALIZED VIEW view_name
AS
SELECT ...
FROM source_table;
```

##### Incremental Refresh and Cost-Based Optimization

One important capability of Materialized Views is incremental refresh.

Suppose the upstream Silver table contains 10 million records, and only 1,000 new records arrive.

A full recomputation would potentially require processing the entire dataset again.

However, when incremental refresh is supported and selected, Databricks can process the relevant changes and update the Materialized View without rebuilding everything.

**Important concepts:**
- Incremental refresh is not guaranteed for every query.
- The optimizer determines whether incremental refresh is applicable.
- Databricks uses cost-based optimization to choose an appropriate refresh strategy.
- Incremental refresh capabilities depend on supported query patterns, source types, and compute configuration.
- Serverless compute is relevant to the incremental refresh capabilities described in this course.

#### Temporary Views

* Temporary Views are logical representations of SQL queries that exist only within the pipeline's execution context.
* They do not physically store data and are primarily used to simplify intermediate transformations.
* When should we use Temporary Views?
    - You need intermediate transformation logic.
    - You want to simplify complex SQL queries.
    - A dataset is needed only within the pipeline.
    - You do not need the dataset to be available as a persistent Unity Catalog object.

Syntax:
```
CREATE TEMPORARY VIEW view_name AS

SELECT ...
FROM source_table;
```

#### Views 

* A View is a logical representation of a SQL query.
* Ordinary Views do not physically store the query results. Instead, the underlying query is executed when the View is queried.
* It is a persistent catalog object.
* View Limitations in Declarative Pipelines:
    - The pipeline must be a Unity Catalog pipeline.
    - Ordinary Views cannot be defined as streaming queries.
    - Ordinary Views cannot be used as streaming sources for a pipeline.

Syntax:
```
CREATE VIEW catalog.schema.view_name AS

SELECT ...
FROM source_table;
```

### 2. The Declarative Pipeline Graph

* One of the most powerful features of Databricks Declarative Pipelines is automatica dependency resolution.
* Declarative Pipelines use the dependencies defined in SQL or Python code to construct a Pipeline Graph automatically.
* This graph represents:
    - Datasets within the pipeline.
    - Dependencies between datasets.
    - The flow of data between transformations.
    - The execution order determined by those dependencies.


In [0]:
%sql

CREATE SCHEMA bikesales.`bronze`;